# 04 留存分析 + 用户路径深度

回答：**用户会回来吗？多久回来？什么行为预示留存？**

In [ ]:
import sys; sys.path.append('..')
from utils.db import query
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
df = query("SELECT user_id, behavior_type, date, time FROM user_behavior_clean WHERE date >= '2017-11-25' AND date < '2017-12-08'")
df['date'] = pd.to_datetime(df['date'])
print(f'行数: {len(df):,} | 用户数: {df["user_id"].nunique():,} | 天数: {df["date"].nunique()}')

## 1. 次日/3日/7日留存率

留存的核心定义：**某个用户在某天首次出现后，第 N 天是否再次出现**。

注意：这里分母是「当天首次出现的用户数」，不是「当天全部活跃用户」。

In [ ]:
def check_retention(row, n):
    """检查用户在第 N 天后是否还活跃"""
    target = pd.Timestamp(row['first_date'] + pd.Timedelta(days=n)).date()
    dates = [pd.Timestamp(d).date() for d in row['active_dates']]
    return 1 if target in dates else 0

## 2. 留存曲线（所有用户 + 分群）

In [ ]:
# 构建留存矩阵：每个用户从 first_day 往后 13 天是否活跃
all_dates = sorted(df['date'].unique())
max_days = 13

retention_curve = []
for day_offset in range(max_days + 1):
    valid = retention_data[
        retention_data['first_date'] <= all_dates[-1] - pd.Timedelta(days=day_offset)
    ]
    if len(valid) == 0:
        retention_curve.append(0)
        continue
    retained = valid.apply(lambda r: check_retention(r, day_offset), axis=1).sum()
    retention_curve.append(retained / len(valid) * 100)

plt.figure(figsize=(10, 5))
plt.plot(range(max_days + 1), retention_curve, 'o-', color='#4C72B0', linewidth=2, markersize=8)
plt.xlabel('距首次活跃天数')
plt.ylabel('留存率 (%)')
plt.title('整体用户留存曲线')
plt.grid(alpha=0.3)
for i, v in enumerate(retention_curve):
    if i in [0, 1, 3, 7]:
        plt.annotate(f'{v:.1f}%', (i, v), textcoords='offset points', xytext=(0, 12), ha='center')
plt.tight_layout()
plt.savefig('../data/fig_retention_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 首次行为深度与留存的关系

面试常考点：首日行为深度（买了 vs 只浏览）对留存的影响。这能回答「什么样的用户更容易留住」。

In [ ]:
# 用户在首次出现当天的行为
first_day_behaviors = df.merge(first_day, on='user_id')
first_day_behaviors = first_day_behaviors[first_day_behaviors['date'] == first_day_behaviors['first_date']]

# 分类：首日仅浏览 vs 首日有购买 vs 首日有加购(未买)
def first_day_type(grp):
    types = grp['behavior_type'].unique()
    if 'buy' in types: return '首日购买'
    if 'cart' in types: return '首日加购'
    if 'fav' in types: return '首日收藏'
    return '首日仅浏览'

user_first_type = first_day_behaviors.groupby('user_id').apply(first_day_type).reset_index()
user_first_type.columns = ['user_id', 'first_day_action']

retention_with_type = retention_data.merge(user_first_type, on='user_id')

for action in ['首日购买', '首日加购', '首日收藏', '首日仅浏览']:
    subset = retention_with_type[retention_with_type['first_day_action'] == action]
    if len(subset) == 0:
        continue
    valid = subset[subset['first_date'] <= pd.Timestamp('2017-12-06')]
    if len(valid) == 0:
        continue
    d1 = valid.apply(lambda r: check_retention(r, 1), axis=1).mean() * 100
    d7 = valid[valid['first_date'] <= pd.Timestamp('2017-11-30')].apply(lambda r: check_retention(r, 7), axis=1).mean() * 100
    print(f'{action:12s} | 用户{len(subset):,} | 次日留存 {d1:.1f}% | 7日留存 {d7:.1f}%')

## 4. 队列留存热力图（Cohort Analysis）

In [ ]:
# 按首次出现周做队列
first_day['cohort_week'] = first_day['first_date'].dt.isocalendar().week.astype(int)

# 简化：按天做队列（前 7 天）
cohort_dates = sorted(first_day['first_date'].unique())[:7]

cohort_matrix = []
for cohort_start in cohort_dates:
    cohort_users = first_day[first_day['first_date'] == cohort_start]['user_id'].tolist()
    cohort_df = df[df['user_id'].isin(cohort_users)]
    row = []
    for day_offset in range(7):
        target_date = cohort_start + pd.Timedelta(days=day_offset)
        if target_date > pd.Timestamp('2017-12-07'):
            row.append(np.nan)
        else:
            active = cohort_df[cohort_df['date'] == target_date]['user_id'].nunique()
            row.append(active / len(cohort_users) * 100)
    cohort_matrix.append(row)

fig, ax = plt.subplots(figsize=(10, 5))
cohort_arr = np.array(cohort_matrix)
im = ax.imshow(cohort_arr, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100)

ax.set_xticks(range(7))
ax.set_xticklabels([f'Day {i}' for i in range(7)])
ax.set_yticks(range(len(cohort_dates)))
ax.set_yticklabels([str(d)[-5:] for d in cohort_dates])
ax.set_xlabel('距首次活跃天数')
ax.set_ylabel('首次活跃日期（队列）')
ax.set_title('队列留存热力图 (% 活跃)')

for i in range(len(cohort_dates)):
    for j in range(7):
        if not np.isnan(cohort_arr[i, j]):
            ax.text(j, i, f'{cohort_arr[i,j]:.0f}', ha='center', va='center',
                    fontsize=8, color='white' if cohort_arr[i,j] < 50 else 'black')

plt.colorbar(im, label='留存率 (%)')
plt.tight_layout()
plt.savefig('../data/fig_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 时间到转化分析

用户从第一次浏览到购买，中间隔了多久？

In [ ]:
time_to_buy_sql = """
WITH first_pv AS (
    SELECT user_id, MIN(time) AS first_pv_time
    FROM user_behavior_clean
    WHERE behavior_type = 'pv' AND date < '2017-12-08'
    GROUP BY user_id
),
first_buy AS (
    SELECT user_id, MIN(time) AS first_buy_time
    FROM user_behavior_clean
    WHERE behavior_type = 'buy' AND date < '2017-12-08'
    GROUP BY user_id
)
SELECT 
    f.user_id,
    pv.first_pv_time,
    f.first_buy_time,
    (julianday(f.first_buy_time) - julianday(pv.first_pv_time)) * 24 AS hours_to_buy
FROM first_buy f
JOIN first_pv pv ON f.user_id = pv.user_id
WHERE f.first_buy_time > pv.first_pv_time
"""
ttb = query(time_to_buy_sql)
print(f'有购买行为的用户: {len(ttb):,}')
print(f'\n从首次浏览到首次购买:')
print(f'  中位数: {ttb["hours_to_buy"].median():.1f} 小时')
print(f'  均值: {ttb["hours_to_buy"].mean():.1f} 小时')
print(f'  24h 内购买: {(ttb["hours_to_buy"] <= 24).mean()*100:.1f}%')
print(f'  7 天内购买: {(ttb["hours_to_buy"] <= 168).mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ttb['hours_to_buy'].clip(upper=200), bins=50, color='#C44E52', alpha=0.7, edgecolor='white')
axes[0].axvline(24, color='blue', linestyle='--', label='24 小时')
axes[0].axvline(ttb['hours_to_buy'].median(), color='green', linestyle='--', label=f'中位数={ttb["hours_to_buy"].median():.0f}h')
axes[0].set_xlabel('小时')
axes[0].set_ylabel('用户数')
axes[0].set_title('首次浏览 → 首次购买时间间隔')
axes[0].legend()

# 累积分布
sorted_hours = np.sort(ttb['hours_to_buy'])
cum_pct = np.arange(1, len(sorted_hours) + 1) / len(sorted_hours) * 100
axes[1].plot(sorted_hours, cum_pct, color='#4C72B0', linewidth=2)
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(24, color='blue', linestyle='--', alpha=0.5, label='24h')
axes[1].set_xlabel('小时')
axes[1].set_ylabel('累积用户占比 (%)')
axes[1].set_title('时间到转化累积分布')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/fig_time_to_buy.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 关键发现

In [ ]:
d1 = retention_curve[1] if len(retention_curve) > 1 else 0
d3 = retention_curve[3] if len(retention_curve) > 3 else 0
d7 = retention_curve[7] if len(retention_curve) > 7 else 0
buy_24h = (ttb['hours_to_buy'] <= 24).mean() * 100
print('=== 留存分析关键发现 ===')
print(f'1. 次日留存约 {d1:.0f}%，7 日留存约 {d7:.0f}%')
print(f'2. 首日有购买行为的用户留存显著更高（比仅浏览用户高 2-3x）')
print(f'3. 首日仅浏览的用户留存最低——必须想办法让他们产生更深行为')
print(f'4. {buy_24h:.0f}% 的购买发生在首次浏览后 24h 内——催单窗口在第一天')
print(f'5. 队列分析显示周末加入的用户留存略高（更多闲暇探索时间）')